# Din Allah Media Engine — first real Wan2.2 video

This notebook is deliberately **guarded**. It checks GPU VRAM and free disk before downloading anything. It never commits model weights or generated video to Git.

Primary candidate: `Wan-AI/Wan2.2-TI2V-5B`. For 24 GB+ VRAM we use the official runtime. For smaller GPUs, we try the maintained DiffSynth-Studio low-VRAM path with CPU/disk offload. If neither path is viable, the notebook stops before burning compute on a doomed run.

In [ ]:
# Cell 1 — preflight only
import os, shutil, subprocess, json, pathlib

print('=== GPU ===')
subprocess.run(['nvidia-smi'], check=False)

free_gb = shutil.disk_usage('/kaggle/working').free / (1024**3)
print(f'Free disk: {free_gb:.1f} GiB')
assert free_gb >= 45, 'STOP: need at least 45 GiB free before model download.'

q = subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader,nounits'], text=True)
rows = [r.strip() for r in q.splitlines() if r.strip()]
print('GPU rows:', rows)
assert rows, 'STOP: no NVIDIA GPU detected.'
gpu_name, vram_mb, driver = [x.strip() for x in rows[0].split(',')]
vram_gb = int(vram_mb) / 1024
print(f'Primary GPU: {gpu_name} / {vram_gb:.1f} GiB / driver {driver}')
print('Mode:', 'official' if vram_gb >= 24 else 'low-vram offload')


In [ ]:
# Cell 2 — install the selected runtime after preflight
import subprocess, sys

subprocess.run([sys.executable,'-m','pip','install','-q','huggingface_hub[cli]'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','ffmpeg-python'], check=True)

if vram_gb < 24:
    subprocess.run([sys.executable,'-m','pip','install','-q','git+https://github.com/modelscope/DiffSynth-Studio.git'], check=True)
else:
    subprocess.run(['git','clone','--depth','1','https://github.com/Wan-Video/Wan2.2.git','/kaggle/working/Wan2.2'], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','-r','/kaggle/working/Wan2.2/requirements.txt'], check=True)


In [ ]:
# Cell 3 — fixed first prompt
PROMPT_ID = 'ant-macro-01'
SEED = 42
PROMPT = (
    'Photorealistic macro documentary footage of a worker ant walking across natural soil and small stones, '
    'visible fine body texture, realistic six-legged gait, shallow depth of field, natural daylight, '
    'stable camera, scientific documentary style, no text, no labels, no people, no logos.'
)
NEGATIVE = 'text, subtitles, logos, watermark, illustration, cartoon, fantasy, deformed insect, extra legs, blurry, static'
OUT = pathlib.Path('/kaggle/working/din-allah-media-run') / PROMPT_ID
OUT.mkdir(parents=True, exist_ok=True)
print(PROMPT_ID, SEED)


In [ ]:
# Cell 4 — generate a short real clip
if vram_gb >= 24:
    import os, subprocess, pathlib
    model_dir = '/kaggle/working/Wan2.2-TI2V-5B'
    if not os.path.isdir(model_dir):
        subprocess.run(['huggingface-cli','download','Wan-AI/Wan2.2-TI2V-5B','--local-dir',model_dir], check=True)
    cmd = [
        'python','/kaggle/working/Wan2.2/generate.py',
        '--task','ti2v-5B','--size','1280*704','--frame_num','49',
        '--ckpt_dir',model_dir,'--offload_model','True','--convert_model_dtype','--t5_cpu',
        '--base_seed',str(SEED),'--prompt',PROMPT
    ]
    subprocess.run(cmd, check=True, cwd='/kaggle/working/Wan2.2')
else:
    # DiffSynth-Studio low-VRAM path: layer/parameter offload to CPU+disk.
    import torch
    from diffsynth.utils.data import save_video
    from diffsynth.pipelines.wan_video import WanVideoPipeline, ModelConfig
    vram_config = {
        'offload_dtype': 'disk', 'offload_device': 'disk',
        'onload_dtype': torch.bfloat16, 'onload_device': 'cpu',
        'preparing_dtype': torch.bfloat16, 'preparing_device': 'cuda',
        'computation_dtype': torch.bfloat16, 'computation_device': 'cuda',
    }
    pipe = WanVideoPipeline.from_pretrained(
        torch_dtype=torch.bfloat16, device='cuda',
        model_configs=[
            ModelConfig(model_id='Wan-AI/Wan2.2-TI2V-5B', origin_file_pattern='models_t5_umt5-xxl-enc-bf16.pth', **vram_config),
            ModelConfig(model_id='Wan-AI/Wan2.2-TI2V-5B', origin_file_pattern='diffusion_pytorch_model*.safetensors', **vram_config),
            ModelConfig(model_id='Wan-AI/Wan2.2-TI2V-5B', origin_file_pattern='Wan2.2_VAE.pth', **vram_config),
        ],
        tokenizer_config=ModelConfig(model_id='Wan-AI/Wan2.1-T2V-1.3B', origin_file_pattern='google/umt5-xxl/'),
    )
    video = pipe(prompt=PROMPT, negative_prompt=NEGATIVE, seed=SEED, tiled=True, height=704, width=1280, num_frames=49)
    save_video(video, str(OUT / f'{PROMPT_ID}.mp4'), fps=24, quality=5)


In [ ]:
# Cell 5 — locate, validate and fingerprint the real output
import hashlib, json, subprocess, pathlib, shutil
videos = list(pathlib.Path('/kaggle/working').rglob('*.mp4'))
videos = [p for p in videos if 'din-allah-media-run' in str(p)]
assert videos, 'No MP4 was generated.'
src = max(videos, key=lambda p: p.stat().st_mtime)
dst = OUT / f'{PROMPT_ID}.mp4'
if src.resolve() != dst.resolve(): shutil.copy2(src, dst)

probe = subprocess.check_output([
    'ffprobe','-v','error','-show_entries',
    'format=duration,size:stream=index,codec_type,codec_name,width,height,r_frame_rate,channels,sample_rate',
    '-of','json',str(dst)
], text=True)
(OUT/'ffprobe.json').write_text(probe, encoding='utf-8')
digest = hashlib.sha256(dst.read_bytes()).hexdigest()
(OUT/'video.sha256').write_text(digest+'  '+dst.name+'\n', encoding='utf-8')
meta = {
  'promptId': PROMPT_ID, 'seed': SEED, 'prompt': PROMPT,
  'gpu': gpu_name, 'vramGiB': vram_gb, 'driver': driver,
  'model': 'Wan-AI/Wan2.2-TI2V-5B',
  'runtimePath': 'official' if vram_gb >= 24 else 'DiffSynth-Studio-low-vram',
  'generatedVideoIsEvidence': False,
  'output': str(dst), 'sha256': digest
}
(OUT/'run-metadata.json').write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding='utf-8')
print('REAL VIDEO:', dst)
print('SHA256:', digest)
print((OUT/'ffprobe.json').read_text())
